# HySpecNet Mamba Qualitative Reconstruction

This notebook loads a small set of meaningful HySpecNet example patches from Google Drive, runs the best HySpecNet Mamba checkpoint, and exports a thesis-ready qualitative figure with:

1. original pseudo-RGB / false-color patch,
2. reconstructed pseudo-RGB / false-color patch,
3. reconstruction error map,
4. original vs reconstructed spectral curve,
5. PSNR, SAM, compression ratio, and size before/after compression.

HSI cubes do not have a native RGB image. The color panels below are three-band composites with robust percentile stretching. Change `RGB_PRESET` to inspect a different spectral composite.

Default checkpoint: `hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt`.


## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
EXAMPLE_ROOT = DRIVE_HSI / 'data/hyspecnet_examples/mamba_qualitative_examples'
CHECKPOINT_PATH = DRIVE_HSI / 'checkpoints/hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt'
OUTPUT_DIR = DRIVE_HSI / 'analysis/hyspecnet_mamba_qualitative'

RGB_PRESETS = {
    # Good default for structure in EnMAP/HySpecNet terrain patches.
    'false_color_swir_nir_red': (150, 100, 50),
    # Slightly lower bands; often less saturated than the default false color view.
    'balanced_false_color': (120, 80, 35),
    # Approximate visible-like low-band composite; not true RGB, but easier to read visually.
    'visible_like_low_bands': (45, 30, 15),
}
RGB_PRESET = 'false_color_swir_nir_red'
RGB_BANDS = RGB_PRESETS[RGB_PRESET]
RGB_PERCENTILES = (1.0, 99.0)
RGB_GAMMA = 1.15
SHOW_RGB_PRESET_GRID = True

# Thesis export settings.
ORIGINAL_BITS_PER_CHANNEL = 16.0
THESIS_SAMPLE_ORDINAL = 0
THESIS_FIGURE_BASENAME = 'mamba_hyspecnet_reconstruction_example'
SAVE_THESIS_PDF = True
SAVE_THESIS_PNG = True

PIXEL_MODE = 'max_error'  # one of: 'max_error', 'center', 'manual', 'manifest'
MANUAL_PIXEL = (64, 64)   # used only when PIXEL_MODE == 'manual'
USE_BITSTREAM = True      # True: model.compress/decompress; False: forward pass
USE_AMP = True
REQUIRE_CUDA = True
FORCE_REINSTALL_ENV = False

print('Example root:', EXAMPLE_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output dir:', OUTPUT_DIR)
print('RGB preset:', RGB_PRESET, RGB_BANDS)


## 2. Mount Drive and Prepare Repo

In [ ]:
import urllib.request

BOOTSTRAP_URL = (
    'https://raw.githubusercontent.com/mhx1467/master-thesis-code/'
    f'{REPO_REF}/scripts/colab_bootstrap.py'
)
exec(urllib.request.urlopen(BOOTSTRAP_URL).read().decode('utf-8'))
sync_repo(REPO_URL, REPO_DIR, REPO_REF)


## 3. Install Dependencies

This cell reuses the working Mamba Colab setup from `hyperview2_mamba_finetune_colab.ipynb`: Torch 2.7.1 with CUDA 12.6 plus prebuilt `causal-conv1d` and `mamba-ssm` wheels. On the first run it intentionally restarts the runtime after installing binary modules. After reconnect, rerun from the repo cell; the marker file makes this cell skip reinstalling.


In [ ]:
install_mamba_env(
    marker_name='.hsi_compression_hyspecnet_qual_env_v2_torch27_mamba232',
    force_reinstall=FORCE_REINSTALL_ENV,
    require_cuda=REQUIRE_CUDA,
    extra_packages=('eotdl', 'tqdm', 'matplotlib', 'ipywidgets'),
    error_message=(
        'mamba-ssm is required for this notebook. Use a fresh GPU Colab runtime, '
        'set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ),
)


## 4. Load Example Patch Manifest

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

manifest_path = EXAMPLE_ROOT / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing example manifest: {manifest_path}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

manifest = json.loads(manifest_path.read_text())
for name, bands in manifest.get('rgb_presets', {}).items():
    RGB_PRESETS[name] = tuple(int(v) for v in bands)
if RGB_PRESET not in RGB_PRESETS:
    raise KeyError(f'Unknown RGB_PRESET={RGB_PRESET}. Available: {sorted(RGB_PRESETS)}')
RGB_BANDS = RGB_PRESETS[RGB_PRESET]

samples = manifest['samples']
print('Examples:', len(samples))
print('Protocol:', manifest.get('dataset_protocol', 'n/a'))
print('RGB preset:', RGB_PRESET, RGB_BANDS)

sample_table = pd.DataFrame(samples)
preferred_cols = [
    'split', 'split_index', 'id', 'file', 'shape', 'min', 'max', 'mean', 'std',
    'selection_reason',
]
display(sample_table[[col for col in preferred_cols if col in sample_table.columns]])


## 5. Build and Load the Mamba Model

In [ ]:
import math
from collections import OrderedDict

import torch

from hsi_compression.models.registry import build_model

ENTROPY_RUNTIME_KEYS = (
    'entropy_bottleneck._offset',
    'entropy_bottleneck._quantized_cdf',
    'entropy_bottleneck._cdf_length',
)


def load_checkpoint_model(checkpoint_path: Path, in_channels: int, device: torch.device):
    raw = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = raw.get('config', {})
    model_section = cfg.get('model', {})
    model_name = model_section.get('model_name')
    if not model_name:
        raise ValueError('Checkpoint does not contain config.model.model_name')
    kwargs = dict(model_section.get('model_kwargs', {}))
    kwargs.pop('in_channels', None)

    model = build_model(model_name=model_name, in_channels=in_channels, **kwargs).to(device)
    state = raw['model_state_dict']
    filtered = OrderedDict(
        (key, value)
        for key, value in state.items()
        if not any(key == runtime_key for runtime_key in ENTROPY_RUNTIME_KEYS)
    )
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    unexpected = list(unexpected)
    not_runtime_missing = [key for key in missing if not any(key == r for r in ENTROPY_RUNTIME_KEYS)]
    if unexpected or not_runtime_missing:
        raise RuntimeError(f'Unexpected load_state_dict result: missing={missing}, unexpected={unexpected}')
    if hasattr(model, 'update'):
        model.update(force=True)
    model.eval()
    return model, cfg, raw

first_arr = np.load(EXAMPLE_ROOT / samples[0]['file'])
in_channels = int(first_arr.shape[0])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg, raw_checkpoint = load_checkpoint_model(CHECKPOINT_PATH, in_channels, device)

print('Device:', device)
print('Model:', cfg.get('model', {}).get('model_name'))
print('Experiment:', cfg.get('experiment', {}).get('name'))
print('Checkpoint epoch:', raw_checkpoint.get('epoch'))
print('Best val loss:', raw_checkpoint.get('best_val_loss'))


## 6. Reconstruction and Plot Helpers

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Serif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
})


def bytes_to_human(num_bytes: int | float | None) -> str:
    if num_bytes is None:
        return 'n/a'
    value = float(num_bytes)
    for unit in ('B', 'KiB', 'MiB', 'GiB'):
        if abs(value) < 1024.0 or unit == 'GiB':
            return f'{value:.2f} {unit}' if unit != 'B' else f'{value:.0f} {unit}'
        value /= 1024.0
    return f'{value:.2f} GiB'


def bitstream_num_bytes(strings) -> int:
    if isinstance(strings, (bytes, bytearray, memoryview)):
        return len(strings)
    if isinstance(strings, str):
        return len(strings.encode('utf-8'))
    if isinstance(strings, (list, tuple)):
        return sum(bitstream_num_bytes(item) for item in strings)
    if isinstance(strings, dict):
        return sum(bitstream_num_bytes(item) for item in strings.values())
    raise TypeError(f'Unsupported bitstream container: {type(strings)!r}')


def cube_to_rgb(
    cube: np.ndarray,
    bands=RGB_BANDS,
    params=None,
    percentile_range=RGB_PERCENTILES,
    gamma=RGB_GAMMA,
):
    channels = []
    if params is None:
        params = []
        for band in bands:
            plane = cube[band]
            lo, hi = np.percentile(plane, percentile_range)
            if hi <= lo:
                hi = lo + 1e-8
            params.append((float(lo), float(hi)))
    for band, (lo, hi) in zip(bands, params):
        channels.append(np.clip((cube[band] - lo) / (hi - lo), 0.0, 1.0))
    rgb = np.stack(channels, axis=-1).astype(np.float32)
    if gamma != 1.0:
        rgb = np.power(np.clip(rgb, 0.0, 1.0), 1.0 / gamma)
    return rgb, params


def choose_pixel(sample: dict, x: np.ndarray, x_hat: np.ndarray):
    if PIXEL_MODE == 'manual':
        row, col = MANUAL_PIXEL
        return int(row), int(col)
    if PIXEL_MODE == 'manifest' and sample.get('spectrum_coords'):
        row, col = sample['spectrum_coords'][0]
        return int(row), int(col)
    if PIXEL_MODE == 'center':
        return x.shape[1] // 2, x.shape[2] // 2
    err = np.mean(np.abs(x_hat - x), axis=0)
    return tuple(int(v) for v in np.unravel_index(np.argmax(err), err.shape))


def metric_summary(x: np.ndarray, x_hat: np.ndarray, compressed_bytes: int | None):
    diff = x_hat - x
    mse = float(np.mean(diff ** 2))
    mae = float(np.mean(np.abs(diff)))
    psnr = float(10.0 * math.log10(1.0 / max(mse, 1e-12)))
    x_flat = x.reshape(x.shape[0], -1).T
    y_flat = x_hat.reshape(x_hat.shape[0], -1).T
    denom = np.linalg.norm(x_flat, axis=1) * np.linalg.norm(y_flat, axis=1)
    valid = denom > 1e-12
    cos = np.ones(x_flat.shape[0], dtype=np.float64)
    cos[valid] = np.sum(x_flat[valid] * y_flat[valid], axis=1) / denom[valid]
    sam = float(np.degrees(np.mean(np.arccos(np.clip(cos, -1.0, 1.0)))))

    original_bytes = int(round(x.size * ORIGINAL_BITS_PER_CHANNEL / 8.0))
    actual_bpppc = None
    compression_ratio = None
    if compressed_bytes is not None and compressed_bytes > 0:
        actual_bpppc = float(compressed_bytes * 8.0 / x.size)
        compression_ratio = float(original_bytes / compressed_bytes)
    return {
        'mse': mse,
        'mae': mae,
        'psnr_db': psnr,
        'sam_deg': sam,
        'original_bytes': original_bytes,
        'compressed_bytes': compressed_bytes,
        'actual_bpppc': actual_bpppc,
        'compression_ratio': compression_ratio,
    }


@torch.no_grad()
def reconstruct_cube(cube: np.ndarray):
    x = torch.from_numpy(cube).unsqueeze(0).float().to(device)
    start = time.perf_counter()
    used = 'bitstream' if USE_BITSTREAM else 'forward'
    compressed_bytes = None
    try:
        with torch.autocast(device_type=device.type, enabled=USE_AMP and device.type == 'cuda', dtype=torch.float16):
            if USE_BITSTREAM:
                packed = model.compress(x)
                compressed_bytes = bitstream_num_bytes(packed['strings'])
                decoded = model.decompress(**packed)
                x_hat = decoded['x_hat']
            else:
                x_hat = model(x)['x_hat']
    except Exception as exc:
        print('Bitstream/forward path failed, retrying plain forward. Error:', repr(exc))
        used = 'forward_fallback'
        compressed_bytes = None
        with torch.autocast(device_type=device.type, enabled=False):
            x_hat = model(x)['x_hat']
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    x_hat = x_hat[0].detach().cpu().float().clamp(0.0, 1.0).numpy()
    return x_hat, used, elapsed, compressed_bytes


def plot_panel(sample: dict, x: np.ndarray, x_hat: np.ndarray, metrics: dict, output_base: Path):
    row, col = choose_pixel(sample, x, x_hat)
    rgb_orig, rgb_params = cube_to_rgb(x, bands=RGB_BANDS)
    rgb_recon, _ = cube_to_rgb(x_hat, bands=RGB_BANDS, params=rgb_params)
    spatial_mae = np.mean(np.abs(x_hat - x), axis=0)

    fig = plt.figure(figsize=(6.9, 5.1), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.92], width_ratios=[1.0, 1.0, 0.88])
    ax_orig = fig.add_subplot(gs[0, 0])
    ax_recon = fig.add_subplot(gs[0, 1])
    ax_error = fig.add_subplot(gs[0, 2])
    ax_spectrum = fig.add_subplot(gs[1, 0:2])
    ax_stats = fig.add_subplot(gs[1, 2])

    ax_orig.imshow(rgb_orig)
    ax_orig.set_title('(a) Oryginał')
    ax_orig.axis('off')

    ax_recon.imshow(rgb_recon)
    ax_recon.set_title('(b) Rekonstrukcja')
    ax_recon.axis('off')

    error_vmax = float(np.percentile(spatial_mae, 99.0))
    im0 = ax_error.imshow(spatial_mae, cmap='magma', vmin=0.0, vmax=max(error_vmax, 1e-8))
    ax_error.set_title('(c) Błąd MAE')
    ax_error.axis('off')
    cbar = fig.colorbar(im0, ax=ax_error, fraction=0.046, pad=0.02)
    cbar.ax.tick_params(labelsize=7)

    band_axis = np.arange(x.shape[0])
    ax_spectrum.plot(band_axis, x[:, row, col], color='black', linewidth=1.7, label='oryginał')
    ax_spectrum.plot(
        band_axis,
        x_hat[:, row, col],
        color='#c44e52',
        linewidth=1.45,
        linestyle='--',
        label='rekonstrukcja',
    )
    for band, color in zip(RGB_BANDS, ('#c44e52', '#55a868', '#4c72b0')):
        ax_spectrum.axvline(band, color=color, linewidth=0.7, alpha=0.35)
    ax_spectrum.set_title(f'(d) Widmo piksela ({row}, {col})')
    ax_spectrum.set_xlabel('Indeks pasma')
    ax_spectrum.set_ylabel('Znormalizowane odbicie')
    ax_spectrum.grid(alpha=0.22, linewidth=0.6)
    ax_spectrum.legend(frameon=False, loc='best')

    ax_stats.axis('off')
    stats_lines = [
        ('PSNR', f"{metrics['psnr_db']:.2f} dB"),
        ('SAM', f"{metrics['sam_deg']:.2f}°"),
        ('CR', f"{metrics['compression_ratio']:.1f}:1" if metrics['compression_ratio'] else 'n/a'),
        ('bpppc', f"{metrics['actual_bpppc']:.4f}" if metrics['actual_bpppc'] else 'n/a'),
        ('Przed', bytes_to_human(metrics['original_bytes'])),
        ('Po', bytes_to_human(metrics['compressed_bytes'])),
    ]
    y = 0.95
    ax_stats.text(0.0, y, '(e) Metryki', weight='bold', fontsize=10, transform=ax_stats.transAxes)
    y -= 0.14
    for label, value in stats_lines:
        ax_stats.text(0.0, y, label, color='0.25', transform=ax_stats.transAxes)
        ax_stats.text(1.0, y, value, ha='right', weight='semibold', transform=ax_stats.transAxes)
        y -= 0.115
    ax_stats.text(
        0.0,
        0.03,
        f"easy/test #{sample.get('split_index', 'n/a')}\nRGB: {RGB_BANDS}",
        fontsize=7.5,
        color='0.35',
        transform=ax_stats.transAxes,
    )

    if SAVE_THESIS_PNG:
        png_path = output_base.with_suffix('.png')
        fig.savefig(png_path, bbox_inches='tight')
        print('Saved:', png_path)
    if SAVE_THESIS_PDF:
        pdf_path = output_base.with_suffix('.pdf')
        fig.savefig(pdf_path, bbox_inches='tight')
        print('Saved:', pdf_path)
    display(fig)
    plt.close(fig)


def plot_rgb_preset_grid(sample: dict, x: np.ndarray, x_hat: np.ndarray, output_path: Path):
    presets = list(RGB_PRESETS.items())
    fig, axes = plt.subplots(len(presets), 2, figsize=(8.5, 3.8 * len(presets)))
    if len(presets) == 1:
        axes = np.array([axes])
    for row_idx, (name, bands) in enumerate(presets):
        rgb_orig, params = cube_to_rgb(x, bands=bands)
        rgb_recon, _ = cube_to_rgb(x_hat, bands=bands, params=params)
        axes[row_idx, 0].imshow(rgb_orig)
        axes[row_idx, 0].set_title(f'Original | {name} | bands={bands}')
        axes[row_idx, 0].axis('off')
        axes[row_idx, 1].imshow(rgb_recon)
        axes[row_idx, 1].set_title(f'Reconstruction | {name}')
        axes[row_idx, 1].axis('off')
    fig.suptitle(f"Color composites for easy/test index={sample.get('split_index', 'n/a')}", fontsize=12)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    display(fig)
    plt.close(fig)


## 7. Run Reconstruction for All Example Patches

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for index, sample in enumerate(samples):
    path = EXAMPLE_ROOT / sample['file']
    print(f"[{index + 1}/{len(samples)}] Loading {path.name}")
    x = np.load(path).astype(np.float32)
    if x.shape[0] != in_channels:
        raise ValueError(f"Unexpected channel count for {path}: {x.shape}")
    x_hat, mode_used, elapsed, compressed_bytes = reconstruct_cube(x)
    metrics = metric_summary(x, x_hat, compressed_bytes=compressed_bytes)
    metrics.update({
        'sample_id': sample['id'],
        'split_index': sample.get('split_index'),
        'mode_used': mode_used,
        'time_sec': elapsed,
        'rgb_preset': RGB_PRESET,
        'rgb_bands': str(RGB_BANDS),
        'original_size': bytes_to_human(metrics['original_bytes']),
        'compressed_size': bytes_to_human(metrics['compressed_bytes']),
    })
    rows.append(metrics)

    out_base = OUTPUT_DIR / f"{index:02d}_{sample['id']}_mamba_panel"
    plot_panel(sample, x, x_hat, metrics, out_base)

    if index == THESIS_SAMPLE_ORDINAL:
        thesis_base = OUTPUT_DIR / THESIS_FIGURE_BASENAME
        plot_panel(sample, x, x_hat, metrics, thesis_base)
        print('Thesis figure basename:', thesis_base)

    if SHOW_RGB_PRESET_GRID:
        out_grid = OUTPUT_DIR / f"{index:02d}_{sample['id']}_color_presets.png"
        plot_rgb_preset_grid(sample, x, x_hat, out_grid)
        print('Saved:', out_grid)

    np.savez_compressed(
        OUTPUT_DIR / f"{index:02d}_{sample['id']}_reconstruction.npz",
        original=x,
        reconstruction=x_hat,
    )

summary = pd.DataFrame(rows)
summary_path = OUTPUT_DIR / 'summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
display(summary)


## 8. Optional Interactive Spectrum Browser

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    recon_files = sorted(OUTPUT_DIR.glob('*_reconstruction.npz'))
    if not recon_files:
        raise FileNotFoundError('No reconstruction files found. Run the previous cell first.')

    dropdown = widgets.Dropdown(
        options=[(path.name, str(path)) for path in recon_files],
        description='sample',
        layout=widgets.Layout(width='900px'),
    )
    row_slider = widgets.IntSlider(value=64, min=0, max=127, step=1, description='row')
    col_slider = widgets.IntSlider(value=64, min=0, max=127, step=1, description='col')

    def show(path_str, row, col):
        clear_output(wait=True)
        display(widgets.VBox([dropdown, row_slider, col_slider]))
        data = np.load(path_str)
        x = data['original']
        x_hat = data['reconstruction']
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(x[:, row, col], color='black', linewidth=2.0, label='original')
        ax.plot(x_hat[:, row, col], color='#d55e00', linewidth=1.6, label='reconstruction')
        ax.set_title(f'Spectrum at pixel ({row}, {col})')
        ax.set_xlabel('Band')
        ax.set_ylabel('Normalized reflectance')
        ax.grid(alpha=0.25)
        ax.legend()
        plt.show()

    ui = widgets.interactive_output(show, {'path_str': dropdown, 'row': row_slider, 'col': col_slider})
    display(widgets.VBox([dropdown, row_slider, col_slider]), ui)
except Exception as exc:
    print('Interactive browser unavailable:', repr(exc))
